In [ ]:
# ── PHASE 3, RUN 5 ───────────────────────────────────────────────────────
# Paste as the ONLY cell in a fresh Kaggle notebook.
# Save Version -> Save & Run All (Commit). Do not run interactively.
# Accelerator must be GPU T4 x2 (set in the notebook sidebar; API pushes get a
# P100, which Kaggle's PyTorch no longer supports).
#
# Same experiment as run 4 - 1536 tokens, no few-shot, 2,500 questions - with
# the run made safe against what killed run 4:
#
#   watchdog          Run 4 loaded the model and then produced nothing for
#                     seven hours, which burned the week's GPU quota. The cell
#                     now watches the phase-3 log; 25 minutes without a new
#                     line kills the process and retries ONCE with a smaller
#                     batch and the default CUDA allocator. Shards written so
#                     far are resumed, not regenerated.
#
#   stack dumps       run_phase3.py writes every thread's stack to the log
#                     every 15 minutes while stalled, so a hang says where.
#
#   per-batch log     Every batch logs its wall time. "Nothing for an hour"
#                     is now visible after five minutes.
#
#   5-hour budget     Kaggle currently allows this account 6 GPU hours a week.
#                     Generation stops itself at 5 hours across both attempts,
#                     flushes, consolidates and saves. ~2,400 questions at the
#                     measured 7.3 s/question.
#
# Expected: overall accuracy 44.3% -> 46-48%.

import glob
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

REPO = 'https://github.com/KJSK-Koushik/Adaptive-reasoning-for-Financial-Advisory-Systems.git'
WORK, INPUT = Path('/kaggle/working'), Path('/kaggle/input')
PROJECT, FRESH = WORK / 'project', WORK / '_repo'

os.chdir(WORK)

# ── 1. data ──────────────────────────────────────────────────────────────
found = glob.glob(str(INPUT / '**' / 'data' / 'processed' / 'unified.parquet'),
                  recursive=True)
if not found:
    raise SystemExit('unified.parquet not found. Sidebar -> Add Input -> your '
                     'Datasets -> adaptive-reasoning-fas, then re-run.')
data_root = Path(found[0]).parent.parent.parent
print('data from:', data_root)

if PROJECT.exists():
    shutil.rmtree(PROJECT)
PROJECT.mkdir(parents=True)
shutil.copytree(data_root / 'data', PROJECT / 'data')

# ── 2. code ──────────────────────────────────────────────────────────────
if FRESH.exists():
    shutil.rmtree(FRESH)
print('cloning the current code ...')
subprocess.check_call(['git', 'clone', '--depth', '1', '--quiet', REPO, str(FRESH)],
                      cwd=str(WORK))
for part in ('src', 'scripts', 'configs'):
    shutil.copytree(FRESH / part, PROJECT / part)
for f in ('requirements.txt', 'requirements-gpu.txt', 'pyproject.toml'):
    if (FRESH / f).exists():
        shutil.copy2(FRESH / f, PROJECT / f)

commit = subprocess.run(['git', '-C', str(FRESH), 'log', '-1', '--format=%h %s'],
                        capture_output=True, text=True, cwd=str(WORK)).stdout.strip()
print('code at commit:', commit)
os.chdir(PROJECT)

# Every feature this run depends on must be in the clone, or five hours produce
# the traces we already have.
cfg_src = Path('src/adaptive_reasoning/config.py').read_text(encoding='utf-8')
runner_src = Path('src/adaptive_reasoning/traces/runner.py').read_text(encoding='utf-8')
phase3_src = Path('scripts/run_phase3.py').read_text(encoding='utf-8')
for needle, where in (('max_wall_seconds', cfg_src),
                      ('MIN_PROBE_AGREEMENT', runner_src),
                      ('batch done', runner_src),
                      ('faulthandler', phase3_src)):
    assert needle in where, f'cloned code is missing {needle} - clone is stale'
print('wall-clock budget: yes | probe guard: yes | per-batch log: yes | stack dumps: yes')

# ── 3. dependencies ──────────────────────────────────────────────────────
try:
    import torch  # noqa: F401
    import transformers  # noqa: F401
    print('dependencies already present')
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           '-r', 'requirements.txt'])

# ── 4. configuration ─────────────────────────────────────────────────────
TOTAL_BUDGET = 5 * 3600        # generation seconds across both attempts
STALL_SECONDS = 25 * 60        # no new log line for this long = hung

def write_config(batch_size, budget_seconds):
    Path('configs/experiment').mkdir(parents=True, exist_ok=True)
    Path('configs/experiment/run5.yaml').write_text(f"""llm:
  model_id: deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
  batch_size: {batch_size}
  max_new_tokens: 1536

traces:
  n_questions: 2500
  step_tokens: 48
  max_steps: 32
  probe_max_tokens: 24
  checkpoint_every: 100
  max_wall_seconds: {int(budget_seconds)}
""")

write_config(32, TOTAL_BUDGET)
print(Path('configs/experiment/run5.yaml').read_text())

# ── 5. gate, then the run, under a watchdog ──────────────────────────────
env = dict(os.environ, PYTHONUNBUFFERED='1')
rc = subprocess.call([sys.executable, 'scripts/run_pilot.py', '--experiment', 'run5'],
                     env=env)
if rc != 0:
    raise SystemExit(f'Pilot gate failed (exit {rc}). Do not start the long run.')

LOG_DIR = PROJECT / 'artifacts' / 'logs'

def newest_log_size():
    logs = sorted(LOG_DIR.glob('phase3-*.log'), key=lambda p: p.stat().st_mtime)
    return logs[-1].stat().st_size if logs else 0

def run_phase3(batch_size, alloc_conf, budget_seconds):
    """Run phase 3; return its exit code, or 'stalled' if the watchdog killed it."""
    write_config(batch_size, budget_seconds)
    run_env = dict(env)
    if alloc_conf:
        run_env['PYTORCH_CUDA_ALLOC_CONF'] = alloc_conf
    else:
        run_env.pop('PYTORCH_CUDA_ALLOC_CONF', None)
    print(f'\n>>> phase 3: batch {batch_size}, allocator '
          f'{alloc_conf or "default"}, budget {budget_seconds/3600:.1f} h')
    proc = subprocess.Popen([sys.executable, 'scripts/run_phase3.py',
                             '--experiment', 'run5'], env=run_env)
    size, changed = newest_log_size(), time.time()
    while proc.poll() is None:
        time.sleep(30)
        now_size = newest_log_size()
        if now_size != size:
            size, changed = now_size, time.time()
        elif time.time() - changed > STALL_SECONDS:
            print(f'\nWATCHDOG: no log output for {STALL_SECONDS // 60} minutes - '
                  f'killing phase 3 (see the stack dumps above for where it sat)')
            proc.kill()
            proc.wait()
            return 'stalled'
    return proc.returncode

gen_started = time.time()
rc = run_phase3(32, 'expandable_segments:True', TOTAL_BUDGET)
if rc == 'stalled':
    remaining = TOTAL_BUDGET - (time.time() - gen_started)
    if remaining > 1800:
        print('retrying once: batch 16, default allocator, resuming from shards')
        rc = run_phase3(16, None, remaining)
    else:
        print('no budget left for a retry')
print('\nexit code:', rc)

# ── 6. copy results to the notebook root ─────────────────────────────────
# Belt and braces: the previous run's /kaggle/working/project never appeared in
# the saved output. Putting the files at the top level makes them easy to find
# and hard to lose.
out_dir = WORK / 'phase3_results'
out_dir.mkdir(exist_ok=True)
for rel in ('artifacts/traces/traces.parquet',
            'artifacts/traces/trace_summary.parquet',
            'artifacts/results/phase3_summary.json',
            'artifacts/results/phase3_pilot.json'):
    src = PROJECT / rel
    if src.exists():
        dest = out_dir / Path(rel).name
        shutil.copy2(src, dest)
        print(f'  saved {dest.name}  ({dest.stat().st_size/1e6:.1f} MB)')

shards = PROJECT / 'artifacts/traces/_shards'
if shards.exists():
    shutil.make_archive(str(WORK / 'shards'), 'zip', shards)
    print(f'  saved shards.zip  ({(WORK / "shards.zip").stat().st_size/1e6:.1f} MB)')

# ── 7. the numbers ───────────────────────────────────────────────────────
summary = PROJECT / 'artifacts/results/phase3_summary.json'
if not summary.exists():
    raise SystemExit(f'No summary written (exit {rc}). Read the error above.')

s = json.loads(summary.read_text())
print('\n' + '=' * 62)
print('  RUN 5 vs THE TRACES WE HAVE')
print('=' * 62)
print(f"  traces               {s['n_traces']:6,}      (was 4,000)")
print(f"  final accuracy       {s['final_accuracy']*100:6.1f}%     (was 44.3%)")
print(f"  correct at any point {s['solvable_fraction']*100:6.1f}%     (was 65.5%)")
print(f"  mean tokens          {s['mean_total_tokens']:6.0f}      (was 533)")
print(f"  mean steps           {s['mean_steps']:6.1f}      (was 11.5)")
print(f"  probe agreement      {s.get('probe_agreement', 0)*100:6.1f}%     (must be above 90)")
print('=' * 62)
print('\n  48% or above means the truncation fix worked.')
print('  Download the phase3_results folder from the Output tab.')
